# GDELT Article Body Scraper

This notebook iterates over the URLs in the cleaned GDELT dataset and attempts to 
extract the body text of each article using HTTP requests and BeautifulSoup HTML 
parsing. For each article, irrelevant HTML elements (scripts, navigation, footers) 
are removed and the first three substantive paragraphs are extracted. This body text 
will later be used alongside article titles to compare the quality of sentiment 
signals extracted by FinBERT from headlines only versus headlines plus body text — 
an experiment designed to quantify headline bias in financial sentiment analysis. 
The resulting dataset is saved to `01_data/processed/gdelt_wti_with_body.csv`.

### Necessary imports

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


### Read processed news sorted by date

In [3]:
df_clean = pd.read_csv("../01_data/processed/gdelt_wti_clean.csv", parse_dates=['datetime'])
print(f"Artículos cargados: {len(df_clean)}")

Artículos cargados: 3290


### Extract paragraphs from news

Discard irrelevant HTML elements such as *script, style, nav, footer, header, aside*, and extract the first 3 paragraphs of each news.


In [4]:
def scrape_article(url, timeout=10):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Eliminar elementos no relevantes
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        
        # Extraer párrafos
        paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 50]
        
        # Primeros 3 párrafos
        body = ' '.join(paragraphs[:3])
        return body
    
    except Exception as e:
        return f"ERROR: {str(e)[:100]}"


In [5]:
# Probar con 5 artículos distintos del dataset
test_urls = df_clean.sample(5, random_state=42)[['title', 'url', 'domain']].reset_index(drop=True)

for _, row in test_urls.iterrows():
    print(f"\nDominio: {row['domain']}")
    print(f"Título: {row['title'][:80]}")
    body = scrape_article(row['url'])
    print(f"Cuerpo ({len(body)} chars): {body[:200]}")
    print("-" * 60)


Dominio: lelezard.com
Título: NOG Announces Third Quarter 2024 Results , Achieves Record Oil Production
Cuerpo (1305 chars): Northern Oil and Gas, Inc. (NYSE: NOG) ("NOG" or "Company") today announced the Company's third quarter results. "During the third quarter we generated record oil volumes and free cash flow despite li
------------------------------------------------------------

Dominio: fxstreet.com
Título: Canadian Dollar steadies as US shutdown freezes economic data
Cuerpo (351 chars): • CAD fell against the USD as markets digest ongoing US government shutdown and suspended economic data releases. • BLS announced Friday'sNFPjobs report is suspended until federal operations resume; U
------------------------------------------------------------

Dominio: londonlovesbusiness.com
Título: Oil opened higher on sanctions and OPEC+ plans , uncertainty lingers  - London B
Cuerpo (0 chars): 
------------------------------------------------------------

Dominio: banklesstimes.com
Título

### Scrapping of all news downloaded

A new column *body* is added with the first 3 paragraphs of each news. 

**Warning**: Some news will fail at scrapping since could be paywalls or dynamic JavaScript.

In [7]:
bodies = []
total = len(df_clean)

for i, row in df_clean.iterrows():
    body = scrape_article(row['url'])
    bodies.append(body)
    
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{total} — {row['domain']}")
    
    time.sleep(0.5)  # Pequeña pausa para no saturar los servidores

df_clean['body'] = bodies

# Estadísticas básicas
successful = df_clean[~df_clean['body'].str.startswith('ERROR') & (df_clean['body'].str.len() > 100)]
print(f"\nTotal artículos: {total}")
print(f"Con cuerpo extraído: {len(successful)} ({len(successful)/total*100:.1f}%)")
print(f"Fallidos o vacíos: {total - len(successful)}")

50/3290 — streetinsider.com
100/3290 — zerohedge.com
150/3290 — finanzen.at
200/3290 — fxstreet.com
250/3290 — businesstimes.com.sg
300/3290 — streetinsider.com
350/3290 — seekingalpha.com
400/3290 — benzinga.com
450/3290 — gulfbusiness.com
500/3290 — insidermonkey.com
550/3290 — tribune.com.pk
600/3290 — menafn.com
650/3290 — livemint.com
700/3290 — ticker.com
750/3290 — menafn.com
800/3290 — albawaba.com
850/3290 — fool.com.au
900/3290 — rttnews.com
950/3290 — forexlive.com
1000/3290 — tmcnet.com
1050/3290 — edmontonjournal.com
1100/3290 — today.az
1150/3290 — morningstar.com
1200/3290 — businesstimes.com.sg
1250/3290 — brecorder.com
1300/3290 — theglobeandmail.com
1350/3290 — menafn.com
1400/3290 — zerohedge.com
1450/3290 — economictimes.indiatimes.com
1500/3290 — brecorder.com
1550/3290 — seekingalpha.com
1600/3290 — fool.ca
1650/3290 — fool.com
1700/3290 — theglobeandmail.com
1750/3290 — investopedia.com
1800/3290 — fxstreet.com
1850/3290 — economy.rs
1900/3290 — zerohedge.com
195

### Filter news bodies

Eliminate bodies that contain errors, paywalls, metadata, etc.

In [14]:
# Quality filter for 04_gdelt_scraper.ipynb
boilerplate_phrases = [
    "Please try using other words",
    "Source: Shutterstock",
    "enable JavaScript",
    "subscribe to continue",
    "sign in to read",
    "cookies to enhance",
    "404",
]

def is_valid_body(text):
    if not isinstance(text, str) or len(text) < 100:
        return False
    if text.startswith('ERROR'):
        return False
    for phrase in boilerplate_phrases:
        if phrase.lower() in text.lower():
            return False
    return True

df_clean['body_valid'] = df_clean['body'].apply(is_valid_body)
print(f"Bodies válidos: {df_clean['body_valid'].sum()}")
print(f"Bodies descartados: {(~df_clean['body_valid']).sum()}")

Bodies válidos: 2600
Bodies descartados: 690


### Diagnostic data

Find errors or non supported characters for CSV.

In [15]:
# Diagnostic — check for dirty characters in body text
import re

issues = []
for i, row in df_clean.iterrows():
    body = str(row['body'])
    if '&amp;' in body or '&nbsp;' in body or '&#' in body:
        issues.append((i, 'HTML entities', body[:100]))
    if '\n' in body or '\r' in body:
        issues.append((i, 'Newlines', body[:100]))
    if re.search(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', body):
        issues.append((i, 'Control characters', body[:100]))

print(f"Articles with issues: {len(issues)}")
for idx, issue_type, sample in issues[:10]:
    print(f"\n[{issue_type}] Row {idx}: {sample}")

Articles with issues: 0


### Clean body characters

Discard characters that break the CSV.

In [10]:
import html

def clean_body(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    # Fix mojibake — re-encode as latin-1 and decode as utf-8
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass  # If it fails, leave the text as is
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['body'] = df_clean['body'].apply(clean_body)
print("Body text cleaned.")
print(df_clean['body'].str.len().describe().round(0))


Body text cleaned.
count     3290.0
mean       813.0
std       1767.0
min          0.0
25%        204.0
50%        536.0
75%        882.0
max      42495.0
Name: body, dtype: float64


### Save data

In [16]:
filename = "gdelt_wti_with_body.csv"
df_clean.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"Saved in 01_data/processed/{filename}")

Saved in 01_data/processed/gdelt_wti_with_body.csv


### Diagnostic for saved CSV

In [17]:
import pandas as pd
import re
import html

# Read the saved CSV
df_saved = pd.read_csv("../01_data/processed/gdelt_wti_with_body.csv")

# Diagnostic
issues = []
for i, row in df_saved.iterrows():
    body = str(row['body'])
    if '&amp;' in body or '&nbsp;' in body or '&#' in body:
        issues.append((i, 'HTML entities', body[:100]))
    if '\n' in body or '\r' in body:
        issues.append((i, 'Newlines', body[:100]))
    if re.search(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', body):
        issues.append((i, 'Control characters', body[:100]))

print(f"Articles with issues: {len(issues)}")
for idx, issue_type, sample in issues[:10]:
    print(f"\n[{issue_type}] Row {idx}: {sample}")

Articles with issues: 0
